In [ ]:
import os
import re
import anndata as ad
import scanpy as sc
import anndata
import pandas as pd

from cell2location import run_colocation
import scvi
import pyro
import torch
import seaborn as sns

%matplotlib inline

/software/conda/users/gd11/cell2loc_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
results_folder = "../../../data/GBM_LEAP_annotations/cell2location/"
ref_run_name = os.path.join(results_folder, "reference_signatures")
run_name = os.path.join(results_folder, "cell2location_map")

In [ ]:
directory_dict = {}
adata_c2l = {}
for f in batch_files_nmf:
    # Load and store the patient
    ad_vis = ad.read_h5ad(f)
    k = re.sub('(.*?)-.*$', 'sp_batch\\1', ad_vis.obs['sample_name'][0])
    adata_c2l[k] = ad_vis

    # Create dictionary
    directory_dict[k] = re.sub('^.*NMF/(.*?)/anndata.*$', '\\1', f)

In [ ]:
factor_range = [10, 12 ,14, 16, 20, 25, 30, 40, 50]

In [ ]:
res_dict = {}
for key in adata_c2l:
    res_dict[key], adata_c2l[key] = run_colocation(
                   adata_c2l[key], model_name='CoLocatedGroupsSklearnNMF',
                   verbose=False, return_all=True,

                   train_args={
                    'n_fact': factor_range, # IMPORTANT: range of number of factors
                    'n_iter': 20000, # maximum number of training iterations

                    'sample_name_col': 'sample_name', # columns in adata_vis.obs that identifies sample

                    'mode': 'normal',
                    'n_type': 'restart', 'n_restarts': 10 # number of training restarts
                   },

                   model_kwargs={'init': 'random', 
                                 'random_state': 0, 'nmf_kwd_args': {'tol': 0.00001}})